# OCONUS NGWPC Hydrofabric Demo
This notebook walks through the following PI-9 acceptance criteria for the OCONUS (Alaska, Hawaii, Puerto Rico/Virgin Islands) domains. This should be run after building OCONUS NHF.

- domains include PRVI, Hawaii, and Alaska
- existence of the same river miles covered in the operational version of the NWM
- ensuring rivers can be represented as a directed acyclic graphs for routing
- connectivity checks
- flowpath and divide statistics (drainage area, length, etc)
- Ensure that hydrofabric fully complies with the HY_Features (WaterML2 Part 3) data model standard as extended to include flowlines that reduce artificial trans-basin transfers of flow.
- POIs should include at minimum all existing operational data assimilation gages as well as existing calibration gages and waterbodies (i.e. lake/reservoir) and their outlet nexuses.
- Every attempt will be made to maximize the NGWPC Hydrofabric such that the number of divides between 3-10 sq. km and verify that routing computational unit lengths (derived from flowpaths and flowlines) are an integer multiple of a 300 m discretization (acceptable range: 250-350 m)

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

In [ ]:
path_ak = Path("../data/ak_nhf_0.1.56.gpkg")
path_hi = Path("../data/hi_nhf_0.1.56.gpkg")
path_prvi =  Path("../data/prvi_nhf_subcase_2.gpkg")

pd.set_option("display.max_columns", None)

## Domains Include Puerto Rico / Virgin Islands, Hawaii, Alaska
To explore all layers of the GPKGs, open GPKGs in QGIS.

In [ ]:
# Explore Alaska
gdf_divides = gpd.read_file(path_ak, layer="divides")
gdf_fp = gpd.read_file(path_ak, layer="divides")
m = gdf_divides.explore(name="divides", color="green")
m = gdf_fp.explore(m=m, name="flowpaths", color="blue")
m

In [ ]:
# Explore Hawaii
gdf_divides = gpd.read_file(path_hi, layer="divides")
gdf_fp = gpd.read_file(path_hi, layer="flowpaths")
m = gdf_divides.explore(name="divides", color="green")
m = gdf_fp.explore(m=m, name="flowpaths", color="blue")
m

In [ ]:
# Explore Puerto Rico / Virgin Islands
gdf_divides = gpd.read_file(path_prvi, layer="divides")
gdf_divides.explore()
m = gdf_divides.explore(name="divides", color="green")
m = gdf_fp.explore(m=m, name="flowpaths", color="blue")
m

## Existence of the same river miles covered in the operational version of the NWM

TODO: open flowpaths for each layer and add to number of river miles in CONUS 

## Ensuring rivers can be represented as a directed acyclic graphs for routing / connectivity checks

TODO @Dylan: DAGs

## Flowpath and divide statistics (drainage area, length, etc)
TODO: Load attribute tables for divides and fp

## Ensure that hydrofabric fully complies with the HY_Features (WaterML2 Part 3) data model standard as extended to include flowlines that reduce artificial trans-basin transfers of flow.
TODO @DYLAN Add text and go to OE link

## POIs should include at minimum all existing operational data assimilation gages as well as existing calibration gages and waterbodies (i.e. lake/reservoir) and their outlet nexuses

### Waterbodies / Lakes
The `lakes` layer was built in NHF to be a 1:1 representation of NWM operational waterbodies. The `lakes` layer retains all data from Hydrofabric 2.2 (Puerto Rico and Hawaii) and LAKEPARM (Alaska).

Where polygons were available (HI and PRVI), lakes were mapped to the most downstream intersecting flowpath. The most downstream flowpath is chosen with the minimum hydrosequence. In Alaska, points were mapped to their nearest flowpath.

In [90]:
def compare_lakes(nhf_path: Path, nwm_path: Path, domain: str, id_field: str):
    """Compare if lakes are present in an NWM source file and an NHF gages layer"""
    gdf_nwm = gpd.read_file(nwm_path)
    gdf_nhf = gpd.read_file(nhf_path, layer="lakes")
    print(f"{domain} NHF lakes: {len(gdf_nhf)}")
    print(f"{domain} NWM lakes: {len(gdf_nwm)}")
    print(f"{domain} lakes COMID in NWM: {len(gdf_nhf.loc[gdf_nhf['lake_id'].isin(gdf_nwm[id_field])])}")
    display(gdf_nhf.head())

Alaska lakes were retrieved from [NWM v3.0.18 LAKEPARM_AK.nc](https://www.nco.ncep.noaa.gov/pmb/codes/nwprod/nwm.v3.0.18/parm/domain_alaska/LAKEPARM_AK.nc) and saved to a GPKG.

In [91]:
compare_lakes(path_ak, "../data/lakes/input/ak_lakeparm.gpkg", "AK", "lake_id" )

AK NHF lakes: 237
AK NWM lakes: 237
AK lakes COMID in NWM: 237


,nhf_lake_id,ref_fp_id,hy_id,fp_id,virtual_fp_id,dn_nex_id,dn_virtual_nex_id,div_id,lake_id,res_id,LkArea,LkMxE,WeirC,WeirL,WeirE,OrificeC,OrificeA,OrificeE,Dam_Length,ifd,reservoir_index_AnA,reservoir_index_Extended_AnA,reservoir_index_GDL_AK,reservoir_index_Medium_Range,reservoir_index_Short_Range,geometry
0,1,810340249,317,171967.0,174539.0,171967.0,174516.0,171967.0,19029000019123,None,8.746114,242.112549,0.4,10.0,239.505161,0.1,1.0,224.729960,10.0,0.9,None,None,None,None,None,POINT (520915.823 1227965.773)
1,2,810434248,318,172875.0,177263.0,172875.0,177189.0,172875.0,19029000007860,None,67.990491,546.495728,0.4,10.0,545.103198,0.1,1.0,537.212199,10.0,0.9,None,None,None,None,None,POINT (424285.366 1327509.657)
2,3,810383934,319,149270.0,149606.0,149270.0,149586.0,149270.0,75004400013260,None,1.180562,65.401436,0.4,10.0,65.124396,0.1,1.0,63.554504,10.0,0.9,None,None,None,None,None,POINT (186823.192 1189116.815)
3,4,810252930,320,149349.0,149843.0,149349.0,149820.0,149349.0,75004400013261,None,1.034069,858.677734,0.4,10.0,856.423169,0.1,1.0,843.647298,10.0,0.9,None,None,None,None,None,POINT (213918.088 1142585.174)
4,5,810200869,321,119155.0,119172.0,119154.0,119164.0,119155.0,75004200016848,None,1.245905,55.758045,0.4,10.0,50.231239,0.1,1.0,18.912681,10.0,0.9,None,None,None,None,None,POINT (307586.841 1121446.442)


Hawaii lakes were extracted from `nwm_lakes.gpkg` provided by OWP in January 2026. The file was clipped to Hawaii state borders. `lake_id` and `newID` are both NHD COMID.

In [ ]:
compare_lakes(path_hi, "../data/lakes/input/nwm_lakes_hi_input.gpkg", "HI", "newID")

Puerto Rico / Virgin Islands lakes were extracted from `nwm_lakes.gpkg` provided by OWP in January 2026. The file was clipped to Puerto Rico/Virgin Islands borders. `lake_id` and `newID` are both NHD COMID.

In [ ]:
compare_lakes(path_prvi, "../data/lakes/input/nwm_lakes_prvi_input.gpkg", "PRVI", "newID")

### Gages
Gages were extracted from routelink and USGS. Routelink files were downloaded from NWM v3.0.18, converted to GPKG in EPSG:4326, and extracted any row with a populated gage ID field. 

If upstream area information was available, gages were matched to flowpaths/divides using it. If it was not available, gages were matched to nearest flowpath. See connectivity columns in tables below (fp_id, virtual_fp_id, dn_nex_id, dn_virtual_nex_id).

In [88]:
def compare_gages(nhf_path: Path, routelink_path: Path, domain: str):
    """Compare if gages are present in routelink and an NHF gages layer"""
    gdf_routelink = gpd.read_file(routelink_path)

    # extract rows with gages
    gdf_routelink["gages"] = gdf_routelink["gages"].str.strip()
    gdf_routelink = gdf_routelink.loc[gdf_routelink["gages"] != ""].copy()

    gdf_gages = gpd.read_file(nhf_path, layer="gages")
    print(f"{domain} NHF gages: {len(gdf_gages)}")
    print(f"{domain} Routelink gages: {len(gdf_routelink)}")
    print(f"{domain} gage ID in Routelink: {len(gdf_gages.loc[gdf_gages['site_no'].isin(gdf_routelink['gages'])])}")
    display(gdf_gages.head())

In [ ]:
# Function used in NHF-builds to extract routelink gages - this is for demonstration purposes only
def append_from_routelink(
    gdf: gpd.GeoDataFrame, routelink: Path, id_col_name: str, shape: Path | None
) -> gpd.GeoDataFrame:
    """Append gages from RouteLink file to GeoDataFrame

    Use ogr2ogr to convert NC file to GPKG and add EPSG:4326 georef i.e. ogr2ogr RouteLink.gpkg RouteLink.nc -t_srs EPSG:4326 -s_srs EPSG:4326

    Parameters
    ----------
    gdf: GeoDataFrame
        Input dataframe to append to
    routelink : Path
        RouteLink file to extract from
    id_col_name: str
        Column to pull from for site_no in RouteLink
    shape: Path | None
        Shapefile to use for clipping
    """
    gages = gpd.read_file(routelink).to_crs(gdf.crs)

    # first get gages only
    gages = gages.loc[gages[id_col_name].str.strip() != ""].copy()

    # then check intersection if requested
    if shape:
        # Get boundary to clip to
        shp = gpd.read_file(shape).to_crs(gdf.crs)
        merged_geom = shp["geometry"].union_all()
        gages = gages.loc[gages["geometry"].intersects(merged_geom), :].copy()

    gages = gages.rename(columns={id_col_name: "site_no"})
    gages["site_no"] = gages["site_no"].str.strip()

    gages = gpd.GeoDataFrame(gages[["geometry", "site_no"]][~gages["site_no"].isin(gdf["site_no"])].copy())
    # logger.info(f"gages: added {len(gages)} gages from RouteLink not already present in dataset") # commetned for missing imports in demonstration
    gages["status"] = "routelink"
    gages = pd.concat([gdf, gages])
    gages["geometry"] = gages["geometry"].force_2d()

    return gages

In [ ]:
# AK
compare_gages(path_ak, Path("../data/gages/routelink/RouteLink_AK_EPSG4326.gpkg"), "AK")

In [89]:
# HI
compare_gages(path_hi, Path("../data/gages/routelink/RouteLink_HI_EPSG4326.gpkg"), "Hawaii")

Hawaii NHF gages: 425
Hawaii Routelink gages: 58
Hawaii gage ID in Routelink: 58


,site_no,status,hy_id,USGS_basin_km2,ref_fp_id,method_fp_to_gage,fp_id,virtual_fp_id,mainstem_virtual_fp_id,segment_order,div_id,dn_nex_id,dn_virtual_nex_id,geometry
0,16010000,USGS-active,1,NaN,80000700000159,nearest_fp,19704.0,20105,20105,0,19704,19704.0,19777,POINT (436067.669 2447657.104)
1,16011000,USGS-discontinued,2,NaN,80000700002925,nearest_fp,19709.0,20207,20207,2,19709,19708.0,20465,POINT (435917.283 2446813.545)
2,16012000,USGS-discontinued,3,NaN,80000700001275,nearest_fp,19698.0,19971,19971,2,19698,19697.0,19938,POINT (435033.001 2447739.395)
3,16013000,USGS-discontinued,4,NaN,80000700002922,nearest_fp,19684.0,20434,20434,0,19684,19683.0,20094,POINT (438005.105 2445913.214)
4,16014000,USGS-discontinued,5,NaN,80000700000650,nearest_fp,13913.0,13930,13930,0,13913,13913.0,13919,POINT (430293.769 2444994.348)


In [ ]:
# PRVI
compare_gages(path_prvi, Path("../data/gages/routelink/RouteLink_PRVI_EPSG4326.gpkg"), "PRVI")

## Every attempt will be made to maximize the NGWPC Hydrofabric such that the number of divides between 3-10 sq. km and verify that routing computational unit lengths (derived from flowpaths and flowlines) are an integer multiple of a 300 m discretization (acceptable range: 250-350 m)
TODO @DYLAN